# Lesson 4.5 — Dashboard Variables

## Learning Objectives
- Create dropdown and multi-select dashboard variables in Dash Studio
- Inject variables into ADQL queries using `${variableName}` syntax
- Create cascading variables (tier depends on application selection)
- Build a fully dynamic dashboard with user-controlled filters

**Reference**: [Dashboard Variables](https://help.splunk.com/en/appdynamics-saas/get-started/26.2.0/dashboards-and-reports/dash-studio/dashboard-variables) | [Variables in ADQL Queries](https://help.splunk.com/en/appdynamics-saas/get-started/26.2.0/dashboards-and-reports/dash-studio/data-binding/variables-in-adql-queries)

## 1. What are Dashboard Variables?

Dashboard variables appear as interactive **filter controls** at the top of the dashboard — dropdown menus, multi-selects, or text inputs.

When a user changes a variable value (e.g., selects a different application from the dropdown), **all widgets that reference `${variableName}` in their queries automatically reload** with the new data.

### Use cases:
- Filter by **application** to reuse the same dashboard for multiple apps
- Filter by **tier** to focus on a specific service
- Filter by **user experience** (SLOW, VERY_SLOW, ERROR) to show only problematic transactions
- Filter by **time window** (in addition to the global time picker)

### How to add a variable:
1. Open dashboard in **Edit** mode
2. Click the **Filter/Variable** icon (funnel icon) in the top toolbar
3. Click **+ Add Variable**
4. Configure:
   - **Name**: the reference name used in `${name}` (no spaces)
   - **Display Name**: what users see as the label
   - **Type**: Dropdown, Multi-Select, Text Input
   - **Options**: Static values, or from ADQL query
   - **Default value**: pre-selected value
5. Click **Save**

In [ ]:
# Document a complete variable configuration for our StoreFront dashboard

dashboard_variables = [
    {
        "name": "selectedApp",
        "display_name": "Application",
        "type": "Dropdown",
        "options_type": "static",
        "options": ["StoreFront", "PaymentService", "InventoryAPI"],
        "default": "StoreFront",
        "used_in": [
            "WHERE applicationName = '${selectedApp}'",
        ],
        "note": "Simple dropdown — select one application at a time"
    },
    {
        "name": "selectedTier",
        "display_name": "Tier",
        "type": "Dropdown",
        "options_type": "static",
        "options": ["All", "WebFrontend", "OrderService", "InventoryAPI"],
        "default": "All",
        "used_in": [
            "AND tierName = '${selectedTier}'  (when not 'All')",
        ],
        "note": "When 'All' is selected, omit the WHERE clause"
    },
    {
        "name": "selectedExperience",
        "display_name": "User Experience",
        "type": "Multi-Select",
        "options_type": "static",
        "options": ["NORMAL", "SLOW", "VERY_SLOW", "ERROR"],
        "default": ["SLOW", "VERY_SLOW", "ERROR"],
        "used_in": [
            "WHERE userExperience IN (${selectedExperience})",
        ],
        "note": "Multi-select — use IN clause in ADQL"
    },
    {
        "name": "topN",
        "display_name": "Top N transactions",
        "type": "Dropdown",
        "options_type": "static",
        "options": ["5", "10", "20", "50"],
        "default": "10",
        "used_in": [
            "LIMIT ${topN}",
        ],
        "note": "Controls how many rows the table widget shows"
    },
]

print("=== Dashboard Variable Definitions ===")
for v in dashboard_variables:
    print(f"\nVariable: ${{{v['name']}}}  [{v['type']}]")
    print(f"  Label   : {v['display_name']}")
    print(f"  Options : {v['options']}")
    print(f"  Default : {v['default']}")
    print(f"  Usage   : {v['used_in'][0]}")
    print(f"  Note    : {v['note']}")

In [ ]:
# Complete dashboard queries using all variables

parameterized_queries = [
    {
        "widget": "Time Series: Transaction count by experience",
        "query": """
SELECT series(eventTimestamp, '1m'), userExperience, count(*)
FROM transactions
WHERE applicationName = '${selectedApp}'
  AND tierName = '${selectedTier}'
"""
    },
    {
        "widget": "Table: Slow transactions matching filters",
        "query": """
SELECT transactionName, responseTime, tierName, userExperience
FROM transactions
WHERE applicationName = '${selectedApp}'
  AND userExperience IN (${selectedExperience})
ORDER BY responseTime DESC
LIMIT ${topN}
"""
    },
    {
        "widget": "Stat: Error count for selected app",
        "query": """
SELECT count(*) AS totalErrors
FROM transactions
WHERE applicationName = '${selectedApp}'
  AND errorOccurred = true
"""
    },
]

print("=== Parameterized Dashboard Queries ===")
for q in parameterized_queries:
    print(f"Widget: {q['widget']}")
    print(f"Query:{q['query']}")
    print()

# Simulate what happens when user selects specific values
import re

user_selections = {
    "selectedApp": "StoreFront",
    "selectedTier": "OrderService",
    "selectedExperience": "'SLOW','VERY_SLOW','ERROR'",  # pre-formatted for IN clause
    "topN": "10",
}

def resolve_variables(query: str, selections: dict) -> str:
    for var, val in selections.items():
        query = re.sub(rf'\${{\s*{re.escape(var)}\s*}}', val, query)
    return query

print("\n=== After user selects: App=StoreFront, Tier=OrderService ===")
resolved = resolve_variables(parameterized_queries[1]['query'], user_selections)
print(resolved.strip())

## 2. Dynamic Variable Options from ADQL

Instead of a hardcoded static list, you can populate variable options dynamically from an ADQL query.

### Example: Dynamic transaction name list

Variable configuration:
- **Name**: `transactionName`
- **Type**: Multi-Select
- **Options source**: ADQL Query
- **ADQL**: `SELECT distinct(transactionName) FROM transactions ORDER BY transactionName ASC`

When the dashboard loads, this query runs and populates the dropdown with every BT that has data in the current time range. If a new BT is discovered, it automatically appears in the filter — no dashboard editing required.

### Example: Cascading variables

1. `${selectedApp}` = Dropdown from `Applications` query type (in Grafana) or static list
2. `${selectedTier}` = Dropdown from `SELECT distinct(tierName) FROM transactions WHERE applicationName = '${selectedApp}'`

When the user changes the application, the tier dropdown automatically refreshes.

In [ ]:
# EXERCISE 4.5: Build a complete parameterized dashboard query set
#
# Scenario: You want a 'Service Health Dashboard' with variables:
#   ${env}      = Dropdown: prod, staging, dev
#   ${service}  = Dropdown: storefront, payment, inventory
#   ${window}   = Dropdown: 1m, 5m, 10m (for series interval)
#
# TASK: Write 3 ADQL queries that use these variables:

print("=== Service Health Dashboard Variables ===")

variables_to_define = [
    {"name": "env",     "type": "Dropdown", "options": ["prod", "staging", "dev"]},
    {"name": "service", "type": "Dropdown", "options": ["storefront", "payment", "inventory"]},
    {"name": "window",  "type": "Dropdown", "options": ["1m", "5m", "10m"]},
]

for v in variables_to_define:
    print(f"  ${{{v['name']}}}  [{v['type']}]: {v['options']}")

print()
print("Dashboard queries (TASK: fill in the ??? placeholders):")
print()

queries_to_complete = [
    {
        "widget": "Time Series: Transaction rate over time",
        "template": """SELECT series(eventTimestamp, '${window}'), userExperience, count(*)
FROM transactions
WHERE applicationName = ???
  AND ???    -- filter by environment using a custom field or naming convention"""
    },
    {
        "widget": "Stat: P95 response time",
        "template": """-- ADQL doesn't have percentile() directly; use this pattern:
SELECT ???
FROM transactions
WHERE applicationName = '${service}'"""
    },
    {
        "widget": "Table: Top 10 errors for selected service",
        "template": """SELECT ???
FROM transactions
WHERE ??? AND ???
ORDER BY ???
LIMIT 10"""
    },
]

for q in queries_to_complete:
    print(f"Widget: {q['widget']}")
    print(q['template'])
    print()

## Summary

- Dashboard variables turn static dashboards into **dynamic, reusable investigation tools**
- Variable types: Dropdown (single-select), Multi-Select, Text Input
- Reference variables in ADQL using `${variableName}`
- Multi-select variables are inserted as comma-separated values for use with `IN` clauses
- Options can be static lists or **dynamically populated from ADQL** queries
- Cascading variables (tier depends on app) create hierarchical filter UIs

---

**Phase 4 Complete!** Proceed to [Phase 5 — Alerting and Operations](../phase-5-alerting-operations/lesson-5.1-health-rules.ipynb)